# 1. Raw images → tracks

1. Fix filenames (optional).
2. Crop a 150 × 150 px region from every movie (dark-frame subtraction + percentile correction).
3. Track the cropped TIFFs in MATLAB (outside Python, see `docs/MATLAB_TRACKING.md`).
4. Load the `*_tracked.mat` files, convert to µm and seconds, clean the tracks, save `instant_df.csv`.

Folders come from `configs/paths.local.yaml`: `raw_experiment_dir`, `cropped_dir`, `dark_frame_dir`.
Settings come from `configs/pipeline.yaml`.

In [1]:
import sys
sys.path.insert(0, "..")  # so `import sptnano` works from this folder

import polars as pl

from sptnano import config

paths = config.load_paths()
settings = config.load_settings()

from sptnano import crop, tracks

print("raw movies:", paths["raw_experiment_dir"])
print("output:    ", paths["cropped_dir"])

raw movies: Z:\mshannon\2026\September\mshannon_cortical77group_9-17-2026
output:     Z:\mshannon\2026\September\mshannon_cortical77group_9-17-2026_analyze


## 1. Fix filenames (optional)

The metadata comes from the filenames, e.g. `loc-GC_type-vent_mol-HTT_geno-20H20S_001.nd2`.
Each condition is one subfolder of `raw_experiment_dir`. Run with `dry_run=True`, read the printout, then `dry_run=False`.

In [ ]:
condition_folder = paths["raw_experiment_dir"] / "<condition folder>"

# crop.insert_suffix_before_number(condition_folder, "_dday-0", dry_run=True)    # Movie_035.nd2 -> Movie_dday-0_035.nd2
# crop.replace_in_filenames(condition_folder, "dox-pos", "dox-neg", dry_run=True)

## 2. Crop

The first pass shows frame 0 of each movie with sliders: put the box on the cell and click save.
When every movie has a box, the second pass crops them all and writes
`<cropped_dir>/data/Condition_<name>/<movie>_cropped.tif`. Re-running skips finished movies.

In [ ]:
crop_settings = settings["crop"]
selector = crop.ROISelectorV2(
    input_directory=str(paths["raw_experiment_dir"]),
    output_directory=str(paths["cropped_dir"]),
    roi_width=crop_settings["roi_width"],
    roi_height=crop_settings["roi_height"],
    dark_frame_subtraction=crop_settings["dark_frame_subtraction"],
    dark_frame_directory=str(paths["dark_frame_dir"]),
    percentile_correction=crop_settings["percentile_correction"],
    pmin=crop_settings["pmin"],
)
selector.process_conditions()

## 3. Track in MATLAB

Open MATLAB, run SLIMfast / evalSPT from `matlab_tracker_dir` on `<cropped_dir>/data` with the settings in
`docs/MATLAB_TRACKING.md`. It writes a `*_tracked.mat` file next to every cropped TIFF.

## 4. Load and clean the tracks

`filter_stubs` drops tracks shorter than 0.6 s, then `pathfixer` splits a track wherever one step is longer
than 1 µm and drops pieces shorter than 0.6 s.

In [ ]:
acquisition = settings["acquisition"]
track_settings = settings["tracks"]

raw_tracks = tracks.load_tracked_mat_files(
    paths["cropped_dir"] / "data", acquisition["pixel_size_um"], acquisition["frame_interval_s"]
)
instant_df, report = tracks.clean_tracks(
    raw_tracks,
    min_track_time_s=track_settings["min_track_time_s"],
    max_jump_um=track_settings["max_jump_um"],
    min_piece_time_s=track_settings["min_piece_time_s"],
    frame_interval_s=acquisition["frame_interval_s"],
)
tracks.track_length_summary(instant_df, acquisition["frame_interval_s"])

In [ ]:
saved_data = paths["cropped_dir"] / "saved_data"
saved_data.mkdir(exist_ok=True)
instant_df.to_csv(saved_data / "instant_df.csv", index=False)
print("Saved", saved_data / "instant_df.csv")